# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [135]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm.notebook import tqdm
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.metrics import accuracy_score

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [136]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['timestamp'] = pd.to_datetime(X['timestamp'])
        X['hour'] = X['timestamp'].dt.hour
        X['weekday'] = X['timestamp'].dt.weekday
        return X.drop(columns=['timestamp'])

In [137]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column='weekday'):
        self.target_column = target_column
        self.encoder = OneHotEncoder(sparse=False)
        self.cat_features = None

    def fit(self, X, y=None):
        features = X.drop(columns=[self.target_column]) if self.target_column in X.columns else X
        self.cat_features = features.select_dtypes(include=['object', 'category']).columns.tolist()
        self.encoder.fit(X[self.cat_features])
        return self

    def transform(self, X):
        X = X.copy()
        encoded_arr = self.encoder.transform(X[self.cat_features])  
        cols = self.encoder.get_feature_names(self.cat_features)
        encoded_df = pd.DataFrame(encoded_arr, columns=cols, index=X.index)
        X = pd.concat([X.drop(columns=self.cat_features), encoded_df], axis=1)
        return X

In [138]:
class TrainValidationTest:
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state

    def split(self, X, y):
        X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=self.test_size, random_state=self.random_state, stratify=y)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train_val, y_train_val, test_size=self.test_size, random_state=self.random_state, stratify=y_train_val)
        return X_train, X_valid, X_test, y_train, y_valid, y_test

In [139]:
df = pd.read_csv('../data/checker_submits.csv')

pipe = Pipeline([
    ('extractor', FeatureExtractor()),
    ('encoder', MyOneHotEncoder('weekday'))
])

df_processed = pipe.fit_transform(df)

y_processed = df_processed['weekday']
X_processed = df_processed.drop(columns=['weekday'])

splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X_processed, y_processed)

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [140]:
class ModelSelection():
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = {}
        
    def choose(self, X_train, y_train, X_valid, y_valid):
        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f"Estimator: {model_name}")

            grid.fit(X_train, y_train)
            best_params = grid.best_params_
            best_score = grid.best_score_


            valid_score = grid.score(X_valid, y_valid)
            self.best_models[model_name] = {
                'params': best_params,
                'valid_score': valid_score,
                'best_model': grid.best_estimator_
            }

            with tqdm(total=len(grid.cv_results_['params']), desc=model_name) as pbar:
                pbar.update(len(grid.cv_results_['params']))

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}\n")

        best_model_name = max(self.best_models, key=lambda x: self.best_models[x]['valid_score'])
        print(f"Classifier with best validation set accuracy: {best_model_name}")
        return best_model_name

    def best_results(self):
        results_list = []
        for model_name, metrics in self.best_models.items():
            results_list.append({
                'model': model_name,
                'params': metrics['params'],
                'valid_score': metrics['valid_score']
            })
        return pd.DataFrame(results_list)

In [141]:
svm_params = [{'kernel': ['linear', 'rbf'],'random_state': [21],'probability': [True]}]
tree_params = {'max_depth': [None, 10, 20], 'class_weight': ['balanced', None]}
rf_params = {'n_estimators': [10, 50], 'max_depth': [None, 10]}

gs_svm = GridSearchCV(estimator=SVC(), param_grid=svm_params, scoring='accuracy', cv=2)
gs_tree = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy', cv=2)
gs_rf = GridSearchCV(estimator=RandomForestClassifier(), param_grid=rf_params, scoring='accuracy', cv=2)


grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: "SVM", 1: "Decision_Tree", 2: "Random_Forest"}


model_selector = ModelSelection(grids, grid_dict)
best_model_name = model_selector.choose(X_train, y_train, X_valid, y_valid)

results_df = model_selector.best_results()
results_df

Estimator: SVM


SVM:   0%|          | 0/2 [00:00<?, ?it/s]

Best params: {'kernel': 'linear', 'probability': True, 'random_state': 21}
Best training accuracy: 0.633
Validation set accuracy score for best params: 0.607

Estimator: Decision_Tree


Decision_Tree:   0%|          | 0/6 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'max_depth': 20}
Best training accuracy: 0.808
Validation set accuracy score for best params: 0.863

Estimator: Random_Forest


Random_Forest:   0%|          | 0/4 [00:00<?, ?it/s]

Best params: {'max_depth': None, 'n_estimators': 50}
Best training accuracy: 0.851
Validation set accuracy score for best params: 0.889

Classifier with best validation set accuracy: Random_Forest


,model,params,valid_score
0,SVM,"{'kernel': 'linear', 'probability': True, 'ran...",0.607407
1,Decision_Tree,"{'class_weight': None, 'max_depth': 20}",0.862963
2,Random_Forest,"{'max_depth': None, 'n_estimators': 50}",0.888889


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [142]:
class Finalize():
    def __init__(self, estimator):
        self.estimator = estimator
        
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        return f'Accuracy of the final model is {accuracy}'
    
    def save_model(self, path):
        joblib.dump(self.estimator, path)

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [143]:
df = pd.read_csv('../data/checker_submits.csv')
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('weekday'))
])
data = preprocessing.fit_transform(df)
data

,numTrials,hour,weekday,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [144]:
spliter = TrainValidationTest()
X = data.drop('weekday', axis=1)
y = data['weekday']
X_train, X_valid, X_test, y_train, y_valid, y_test = spliter.split(X, y)

In [145]:
model_selector = ModelSelection(grids, grid_dict)
best_model_name = model_selector.choose(X_train, y_train, X_valid, y_valid)
results_df = pd.DataFrame(model_selector.best_results())
results_df

Estimator: SVM


SVM:   0%|          | 0/2 [00:00<?, ?it/s]

Best params: {'kernel': 'linear', 'probability': True, 'random_state': 21}
Best training accuracy: 0.633
Validation set accuracy score for best params: 0.607

Estimator: Decision_Tree


Decision_Tree:   0%|          | 0/6 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'max_depth': 20}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.856

Estimator: Random_Forest


Random_Forest:   0%|          | 0/4 [00:00<?, ?it/s]

Best params: {'max_depth': None, 'n_estimators': 50}
Best training accuracy: 0.853
Validation set accuracy score for best params: 0.904

Classifier with best validation set accuracy: Random_Forest


,model,params,valid_score
0,SVM,"{'kernel': 'linear', 'probability': True, 'ran...",0.607407
1,Decision_Tree,"{'class_weight': 'balanced', 'max_depth': 20}",0.855556
2,Random_Forest,"{'max_depth': None, 'n_estimators': 50}",0.903704


In [146]:
best_model = model_selector.best_models[best_model_name]
best_model

{'params': {'max_depth': None, 'n_estimators': 50},
 'valid_score': 0.9037037037037037,
 'best_model': RandomForestClassifier(n_estimators=50)}

In [150]:
best_model = model_selector.best_models[best_model_name]['best_model']
final = Finalize(best_model)

final_score = final.final_score(X_train, y_train, X_test, y_test)
final_score

'Accuracy of the final model is 0.908284023668639'

In [151]:
filename = f'{best_model_name}_{final_score.split()[-1]}.sav'
final.save_model(filename)